# SMS Spam Classification Using NLP

##  Part 1: Introduction

### Student Information
Aviv M 1886

### AI Prompts and Additional Resources

For this project, I used **GitHub Copilot** extensively during the development of the code, as well as **Claude Code** and **ChatGPT** for coding assistance, understanding concepts, and clarifications.

#### Examples of AI Prompts / Tasks Used

- Using AI assistants to generate initial code blocks for different parts of the project, including preprocessing, feature engineering, and the Naive Bayes classifier.
- Some code blocks were initially generated with **Claude Code**, while others were generated or suggested directly by **GitHub Copilot** during development.
- Asking AI to explain specific lines of code and programming concepts.
- Asking AI to help implement and understand **Grid Search and 5-Fold Cross-Validation**.
- Asking AI to improve and clarify comments and explanations in the code.

The generated code was not used as-is. I reviewed it, changed it, and adapted it to my own implementation and the requirements of the assignment.

In addition, I watched the YouTube video **“Naive Bayes FROM SCRATCH in Python (no scikit-learn, just math)”** by **Harry Connor AI** to better understand how to structure and implement the Naive Bayes classifier from scratch.

The implementation shown in the video uses **continuous features**, while my project uses **discrete features**. Therefore, I used the video mainly as a reference for the general structure of the classifier and the implementation process, and then developed my own implementation according to the requirements and the type of features used in my project.

### Problem and Dataset
The goal of this project is to perform a binary classification of SMS messages and predict whether each message is Spam or Ham. For this project, I'll use the following dataset: "Spam SMS Classification Using NLP", from Kaggle, which contains labeled SMS messages classified as Spam or Ham. After loading the data, I'll perform text Feature Engineering, and then implement the Naive Bayes algorithm to classify the messages.
After implementing the algorithm, I'll perform Grid Search with 5-Fold Cross-Validation to find the best alpha based on the average F1 score, then train the final model on the entire training set using this alpha and evaluate it on the test set using F1 score with spam as the positive class.

### Dataset Loading

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/Spam_SMS.csv")  # load the full dataset from CSV

# split ONCE into train/test (80% / 20%)
# random_state=42: same split every time we run this
# stratify=df["Class"]: keeps the same ham/spam ratio in both train and test (dividing proportionally according to "Class" column)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["Class"]
)

# Making the Class to be most right column for better visualization in the dataframe
train_df = train_df[["Message", "Class"]]
test_df = test_df[["Message", "Class"]]

print("Train set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

Train set shape: (4459, 2)
Test set shape: (1115, 2)


In [17]:
print("First 5 rows of the training set:")
display(train_df.head())

First 5 rows of the training set:


,Message,Class
82,Ok i am on the way to home hi hi,ham
1328,Ur balance is now £500. Ur next question is: W...,spam
577,I'm tired of arguing with you about this week ...,ham
656,Tell them the drug dealer's getting impatient,ham
317,Hmmm... Guess we can go 4 kb n power yoga... H...,ham


In [18]:
print("First 5 rows of the test set:")
display(test_df.head())

First 5 rows of the test set:


,Message,Class
3625,No message..no responce..what happend?,ham
4570,At WHAT TIME should i come tomorrow,ham
1192,Come to my home for one last time i wont do an...,ham
673,Get ur 1st RINGTONE FREE NOW! Reply to this ms...,spam
2873,See you there!,ham


In [4]:
# separate the messages (X) from the classes (y)
# no new split here - just use the train/test sets we already made above (columns separation)
X_train, y_train = train_df["Message"], train_df["Class"]
X_test, y_test = test_df["Message"], test_df["Class"]

## Part 2: Feature Engineering

### Preprocessing
Tokenization, lowercasing, stopwords removal, and stemming are preprocessing steps. They clean and prepare the raw text, but they don't turn it into numerical features yet. The actual feature engineering happens in the Bag-of-Words step below.

### Step 1 - Tokenization (Atomic Segmentation)

split each message into individual word tokens using a regex that matches only word characters (`\w+`). Punctuation is not captured by `\w+`, so it's dropped automatically as part of this step.

In [5]:
import re  # regular expressions module, used for tokenizing text

def tokenize(text):
    # \w+ matches runs of letters/digits/underscore (punctuation not captured)
    return re.findall(r"\w+", text)  # returns a list of the matched token strings, e.g. ["Ok", "i", "am", ...]

# .apply() calls tokenize() once for every message in the Series, and collects all the returned lists into a new Series
X_train_tokens = X_train.apply(tokenize)
X_test_tokens = X_test.apply(tokenize)

# X_train_tokens and X_test_tokens are pandas Series, where each element is a list of tokens (not a single string anymore)
X_train_tokens.head()

82            [Ok, i, am, on, the, way, to, home, hi, hi]
1328    [Ur, balance, is, now, 500, Ur, next, question...
577     [I, m, tired, of, arguing, with, you, about, t...
656     [Tell, them, the, drug, dealer, s, getting, im...
317     [Hmmm, Guess, we, can, go, 4, kb, n, power, yo...
Name: Message, dtype: object

### Step 2 - Lowercase

Lowercase every token so that "Free" and "free", for example, will be treated as the same feature instead of being counted as two different features, reducing the number of unique features in the vocabulary.

In [6]:
def lowercase_tokens(tokens):
    return [t.lower() for t in tokens]  # .lower() converts a string to lowercase letters. list comprehension applies it to every token (word) in the list.

# Same X_train_tokens and X_test_tokens, but now all tokens are lowercase
X_train_tokens = X_train_tokens.apply(lowercase_tokens)
X_test_tokens = X_test_tokens.apply(lowercase_tokens)

X_train_tokens.head()

82            [ok, i, am, on, the, way, to, home, hi, hi]
1328    [ur, balance, is, now, 500, ur, next, question...
577     [i, m, tired, of, arguing, with, you, about, t...
656     [tell, them, the, drug, dealer, s, getting, im...
317     [hmmm, guess, we, can, go, 4, kb, n, power, yo...
Name: Message, dtype: object

### Step 3 - Stopwords removal
Remove common words that don't carry much meaning, like "the", "is", "and", etc. reducing the number of unique features in the vocabulary and the dimensions of the Bag-of-Words matrix.

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # importing a known list of English stop words from sklearn

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS] # returns a list of tokens that are not in ENGLISH_STOP_WORDS

# Same X_train_tokens and X_test_tokens, but now all stop words are removed
X_train_tokens = X_train_tokens.apply(remove_stopwords)
X_test_tokens = X_test_tokens.apply(remove_stopwords)

X_train_tokens.head()

82                                [ok, way, home, hi, hi]
1328    [ur, balance, 500, ur, question, sang, uptown,...
577             [m, tired, arguing, week, week, want, ll]
656           [tell, drug, dealer, s, getting, impatient]
317     [hmmm, guess, 4, kb, n, power, yoga, haha, dun...
Name: Message, dtype: object

### Step 4 - Stemming

Reduce each word to its base form (for example: "winning", "won", "wins" → "win"), so different forms of the same word are treated as one feature instead of separate ones, also reducing the dimensions of the Bag-of-Words matrix.

In [8]:
from nltk.stem import PorterStemmer # importing the PorterStemmer class from the nltk.stem module, which is used for stemming words

stemmer = PorterStemmer() # create a stemmer object that can be used to stem words

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]  # returns a list of stemmed tokens. Porter can produce non-dictionary stems (like: "arguing" -> "argu"), but this doesn't hurt classification since the mapping is consistent across train/test

# Same X_train_tokens and X_test_tokens, but now all tokens are stemmed
X_train_tokens = X_train_tokens.apply(stem_tokens)
X_test_tokens = X_test_tokens.apply(stem_tokens)

X_train_tokens.head()

82                                [ok, way, home, hi, hi]
1328    [ur, balanc, 500, ur, question, sang, uptown, ...
577                 [m, tire, argu, week, week, want, ll]
656                  [tell, drug, dealer, s, get, impati]
317     [hmmm, guess, 4, kb, n, power, yoga, haha, dun...
Name: Message, dtype: object

### Featurizer

This is the actual feature engineering step. turning the processed tokens to numeric Bag-of-Words matrix: every unique word across the training messages becomes a feature (column), and each message (row) gets the raw count of how many times each word appears in it.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer # importing the CountVectorizer class from sklearn, which is used to convert a collection of text documents to a matrix of token counts

vectorizer = CountVectorizer(tokenizer=lambda tokens: tokens, preprocessor=lambda tokens: tokens) # create a CountVectorizer object that uses the tokens i created above, instead of the default behavior of tokenizing and preprocessing 
X_train_bow = vectorizer.fit_transform(X_train_tokens) # learns the vocabulary from the training tokens and returns the train Bag-of-Words matrix (counts)
X_test_bow = vectorizer.transform(X_test_tokens) # applies the vocabulary already learned from train to the test tokens (no re-fitting), returning the test Bag-of-Words matrix

X_train_bow = X_train_bow.toarray() # convert the scipy sparse matrix to a dense numpy array - simpler to index/do math on for the Naive Bayes implementation later
X_test_bow = X_test_bow.toarray()   # same conversion for the test matrix

C:\Users\aviv\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [10]:
# displaying the feature engineered data on 2-3 example rows, with actual word names as column headers
print("Train examples:")
display(pd.DataFrame(X_train_bow[:3, 6200:6220], columns=vectorizer.get_feature_names_out()[6200:6220]))  # this range includes "wont", chosen so the example isn't all zeros

print("Test examples:")
display(pd.DataFrame(X_test_bow[:3, 6200:6220], columns=vectorizer.get_feature_names_out()[6200:6220]))  # same range - "wont" is 1 in the test example, proving the counts actually work

Train examples:


,wlcome,wld,wml,wn,wnevr,wnt,wo,woah,wocay,woke,woken,woman,women,won,wondar,wondarful,wonder,wont,woo,woodland
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Test examples:


,wlcome,wld,wml,wn,wnevr,wnt,wo,woah,wocay,woke,woken,woman,women,won,wondar,wondarful,wonder,wont,woo,woodland
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


## Part 3: Naive Bayes Algorithm Implementation
Implementing Multinomial Naive Bayes from scratch, trained on the Count Bag-of-Words features (X_train_bow) and their corresponding labels (y_train) built in Part 2. The algorithm learns the prior probability of each class and the likelihood of each word given a class, which are then used during prediction to calculate the posterior probability and classify each message as spam or ham based on the higher score.

In [11]:
import numpy as np

In [12]:
class NaiveBayes:
    # Class constructor, called when we create a new instance of the class (e.g., model = NaiveBayes(alpha=1.0))
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing parameter
        self.classes = np.array([])  # array of unique class labels (e.g., ["ham", "spam"]), set in fit()
        self.priors = {}  # dictionary to hold prior probabilities for each class (e.g., {"ham": 0.8, "spam": 0.2})
        self.word_counts = {}  # dictionary to hold word counts for each class (e.g., {"ham": np.array([5, 1, 3, 2]), "spam": np.array([1, 4, 6, 3])})
        self.vocabulary_size = 0  # total number of unique words in the vocabulary

        # dictionary mapping each class to an array of per-word likelihoods (with Laplace smoothing), one value per vocabulary word
        # e.g., {"ham": np.array([0.12, 0.08, 0.15, 0.05]), "spam": np.array([0.03, 0.18, 0.11, 0.09])}, where likelihoods["ham"][i] = P(word_i | ham)
        self.likelihoods = {}

    # Method that trains the Naive Bayes model on the training data (X_train, y_train)
    def fit(self, X_train, y_train):
        self.classes = np.unique(y_train) # find the unique class labels in the training data (e.g., ["ham", "spam"]) and store them in self.classes

        # Calculate class priors
        self.priors = {c: np.sum(y_train == c) / len(y_train) for c in self.classes}  # prior probability for each class

        # Calculate word counts for each class
        for c in self.classes:
            class_rows = X_train[y_train == c]  # class_rows will contain only the rows from X_train that belong to class c
            self.word_counts[c] = np.sum(class_rows, axis=0)  # sum the counts of each word across all messages in class c

        # number of unique words (features) in the training data
        self.vocabulary_size = X_train.shape[1]  

        # Calculate likelihoods for each word in each class with Laplace smoothing
        for c in self.classes:
            total_words_in_class = np.sum(self.word_counts[c])  # total word count across all messages in class c
            smoothed_counts = self.word_counts[c] + self.alpha  # NumPy broadcasting: adds alpha to every element of the array at once (no loop needed), e.g. np.array([3, 2, 0]) + 1 -> np.array([4, 3, 1])
            self.likelihoods[c] = smoothed_counts / (total_words_in_class + self.alpha * self.vocabulary_size) # Calculate likelihoods for each word in class c with Laplace smoothing

    # Method that predicts the class labels for the test data (X_test)
    def predict(self, X_test):
        predictions = [] # will contain the predicted class for each test message e.g., ["ham", "ham", "spam", ...]

        # Iterating the rows of the test matrix, where each row is a message represented as a bag-of-words vector
        for row in X_test:
            log_posteriors = {} # will contain the log posterior probability for each class for the current test message, e.g., {"ham": -12.3, "spam": -15.6}
            for c in self.classes: # 2 iterations: one for "ham" and one for "spam"
                # Calculate log posterior probability for class c
                log_prior = np.log(self.priors[c])  # log(P(c))
                log_likelihood = np.sum(row * np.log(self.likelihoods[c]))  # sum of log(P(word_i | c)) for all words in the message
                log_posteriors[c] = log_prior + log_likelihood  # log(P(c | x)) ∝ log(P(c)) + sum(log(P(word_i | c)))

            # Choose the class with the highest log posterior probability
            predicted_class = max(log_posteriors, key=log_posteriors.get) # Select the class with the highest log posterior (e.g., "ham" if log_posteriors["ham"] > log_posteriors["spam"])
            predictions.append(predicted_class) # Append the predicted class for the current test message to the predictions list

        return np.array(predictions)  # return predictions as a NumPy array e.g., np.array(["ham", "ham", "spam", ...])

## Part 6 (Bonus): Hyperparameter Tuning - Grid Search + 5-Fold CV
Testing different values of the alpha hyperparameter to find the one that gives the best F1 score. F1 is the harmonic mean of precision and recall, which balances between them and measures how well the model identifies the positive class. Since this is a binary classification problem with Spam as the positive class, F1 is used as required by the assignment, with Spam chosen as the positive class because it is the class we want to identify and filter out. For each alpha, 5-fold cross-validation is used to train and validate the model on different splits of the training data. The 5 F1 scores are averaged, and the alpha with the highest average F1 score is selected as the best value.

In [13]:
from sklearn.model_selection import KFold  # class used to create folds for cross-validation
from sklearn.metrics import f1_score       # function used to calculate the F1 score
import pandas as pd

alpha_values = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 20.0]  # alpha values we want to test

# create a KFold object with the parameters we want to use when splitting:
kf = KFold(
    n_splits=5,       # divide the training data into 5 folds
    shuffle=True,     # shuffle the rows before creating the folds for optimal ratio of ham/spam in each fold
    random_state=42   # make the shuffle the same every time we run it
)

results = []  # will contain one result for each alpha, e.g. [{"alpha": 0.1, "avg_f1": 0.91}, ...]

# Grid Search: iterate through all alpha values, with Cross-Validation in the inner loop for each alpha
for alpha in alpha_values:
    fold_scores = []  # will contain the F1 score from each of the 5 folds for the current alpha, e.g. [0.91, 0.85, 0.72, 0.83, 0.89] (the average of these 5 scores will be the final score for this alpha)

    # The actual cross-validation loop, where we train and validate the model on each fold
    # kf.split() gives 5 pairs of row-position arrays, each pair contains 4/5 rows for training and 1/5 for validation
    for train_idx, validation_idx in kf.split(X_train_bow):
    
        X_fold_train, X_fold_validation = X_train_bow[train_idx], X_train_bow[validation_idx]
        # select the corresponding rows from X_train_bow using the index positions above

        y_fold_train, y_fold_validation = y_train.iloc[train_idx], y_train.iloc[validation_idx]
        # same thing for y_train, but .iloc is used because y_train is a pandas Series

        model = NaiveBayes(alpha=alpha)  # create a new Naive Bayes model using the current alpha  
        model.fit(X_fold_train, y_fold_train)   # train the model on the 4/5 of the training data for this fold
        preds = model.predict(X_fold_validation)      # predict the classes of the 1/5 of the validation data for this fold

        score = f1_score(y_fold_validation, preds, pos_label="spam") # calculate the F1 score for Spam, which is the positive class
        fold_scores.append(score)   # save this fold's F1 score in the list

    results.append({"alpha": alpha, "avg_f1": sum(fold_scores) / len(fold_scores)}) # after all 5 folds, calculate the average F1 score for this alpha and save it

results_df = pd.DataFrame(results)  # turn the results list into a table
best_row = results_df.loc[results_df["avg_f1"].idxmax()]  # idxmax() finds the row-index with the highest avg_f1, and .loc[] gets that row as a Series e.g., alpha=3.0, avg_f1=0.93

display(results_df)
print("Best alpha:", best_row["alpha"], "with avg F1:", best_row["avg_f1"])

,alpha,avg_f1
0,0.1,0.915828
1,0.5,0.919379
2,1.0,0.924110
3,2.0,0.933621
4,3.0,0.934152
5,4.0,0.924288
6,5.0,0.920995
7,10.0,0.889611
8,20.0,0.836872


Best alpha: 3.0 with avg F1: 0.9341516128299409


## Part 4: Training
Using the best alpha found during Grid Search and 5-Fold CV, the final Naive Bayes model is trained on the entire training set. This allows the model to learn from all available training data before being evaluated on the untouched test set.

In [ ]:
nb_model = NaiveBayes(alpha=best_row["alpha"])   # create the final model using the best alpha found by Grid Search
nb_model.fit(X_train_bow, y_train)   # train the final model on the entire training set

## Part 5: Prediction and Model Quality Evaluation on the Test Set
Using the final trained model, predictions are made on the untouched test set and the model is evaluated using F1 score, with Spam as the positive class, as required by the assignment. The reason for choosing Spam as the positive class was explained in Part 6 above.

In [15]:
preds = nb_model.predict(X_test_bow)  # predict on the test set using the final model
score = f1_score(y_test, preds, pos_label="spam")  # compute the F1 score on the test set

print("First 5 true labels:")
display(y_test.head(5))  # display the first 5 true labels
print()

preds = pd.Series(preds, name="Prediction", index=y_test.index)   
print("First 5 predictions:")
display(preds.head(5))  # display the first 5 predictions to compare with the true labels

print("Final model F1 score on test set:", score)

First 5 true labels:


3625     ham
4570     ham
1192     ham
673     spam
2873     ham
Name: Class, dtype: object


First 5 predictions:


3625     ham
4570     ham
1192     ham
673     spam
2873     ham
Name: Prediction, dtype: object

Final model F1 score on test set: 0.9219858156028369
